<a href="https://colab.research.google.com/github/rwcitek/Medicare-data/blob/main/medicare-data-minimal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Medicare data



From [Doctors and clinicians datasets](https://data.cms.gov/provider-data/search?theme=Doctors%20and%20clinicians) page.

- [National Downloadable File]( https://data.cms.gov/provider-data/dataset/mj5m-pzi6 )
- [Data Dictionary]( https://data.cms.gov/provider-data/sites/default/files/data_dictionaries/physician/DOC_Data_Dictionary.pdf )


From the National Downloadable File URL, we see that the dataset ID is `mj5m-pzi6`.  We can use that to query the CMS API to get the link to the CSV file.



In [58]:
%%capture pkg_output
%%bash
apt-get update
apt-get install -y tree jq


In [28]:
import requests
import pandas as pd


In [29]:
dataset_id = "mj5m-pzi6"
url = f"https://data.cms.gov/provider-data/api/1/metastore/schemas/dataset/items/{dataset_id}"

dataset_id, url

('mj5m-pzi6',
 'https://data.cms.gov/provider-data/api/1/metastore/schemas/dataset/items/mj5m-pzi6')

In [30]:
response = requests.get(url)
response


<Response [200]>

In [31]:
metadata = response.json()
metadata


{'accessLevel': 'public',
 'landingPage': 'https://data.cms.gov/provider-data/dataset/mj5m-pzi6',
 'bureauCode': ['009:38'],
 'issued': '2023-08-17',
 '@type': 'dcat:Dataset',
 'modified': '2026-05-29',
 'released': '2026-06-11',
 'nextUpdateDate': '2026-07-09',
 'keyword': ['Clinicians', 'Quality', 'Location'],
 'contactPoint': {'@type': 'vcard:Contact',
  'fn': 'CMS',
  'hasEmail': 'mailto:QPP@cms.hhs.gov'},
 'publisher': {'@type': 'org:Organization',
  'name': 'Centers for Medicare & Medicaid Services (CMS)'},
 'identifier': 'mj5m-pzi6',
 'description': 'The Doctors and Clinicians national downloadable file is organized such that each line is unique at the clinician/enrollment record/group/address level. Clinicians with multiple Medicare enrollment records and/or single enrollments linking to multiple practice locations are listed on multiple lines.',
 'title': 'National Downloadable File',
 'programCode': ['009:000'],
 'distribution': [{'@type': 'dcat:Distribution',
   'downloadURL

In [32]:
url = metadata["distribution"][0]["downloadURL"]
url


'https://data.cms.gov/provider-data/sites/default/files/resources/52c3f098d7e56028a298fd297cb0b38d_1780085742/DAC_NationalDownloadableFile.csv'

## Download the CSV file


In [36]:
!curl -O {url}


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  795M    0  795M    0     0  50.8M      0 --:--:--  0:00:15 --:--:-- 65.2M


In [39]:
csv_file = url.split("/")[-1]
csv_file


'DAC_NationalDownloadableFile.csv'

In [40]:
!ls -la {csv_file}

-rw-r--r-- 1 root root 834304976 Jun 14 05:38 DAC_NationalDownloadableFile.csv


In [41]:
!wc -l {csv_file}

3364554 DAC_NationalDownloadableFile.csv


## DuckDB: query a CSV file



In [33]:
import duckdb


In [48]:
duckdb.query(f"""
  SELECT *
  FROM read_csv(
    "{csv_file}",
    types={{'Telephone Number': 'VARCHAR'}},
    auto_detect=True
  )
  LIMIT 10
""").show() ;


┌────────────┬────────────┬─────────────────┬────────────────────┬─────────────────────┬──────────────────────┬─────────┬─────────┬─────────┬─────────┬────────┬────────────────────────────────────────────────┬────────────┬────────────┬────────────┬────────────┬──────────────┬──────────┬─────────────────────────┬────────────┬─────────────┬──────────┬──────────┬───────────┬────────────┬─────────┬──────────┬──────────────────┬───────────┬───────────┬───────────────────────────┐
│    NPI     │ Ind_PAC_ID │   Ind_enrl_ID   │ Provider Last Name │ Provider First Name │ Provider Middle Name │  suff   │  gndr   │  Cred   │ Med_sch │ Grd_yr │                    pri_spec                    │ sec_spec_1 │ sec_spec_2 │ sec_spec_3 │ sec_spec_4 │ sec_spec_all │ Telehlth │      Facility Name      │ org_pac_id │ num_org_mem │ adr_ln_1 │ adr_ln_2 │ ln_2_sprs │ City/Town  │  State  │ ZIP Code │ Telephone Number │ ind_assgn │ grp_assgn │          adrs_id          │
│   int64    │  varchar   │     varchar 

## DuckDB: query a CSV file with a CTE


In [49]:
duckdb.query(f"""
  WITH
  medicare as (
    SELECT *
    FROM read_csv(
      "{csv_file}",
      types={{'Telephone Number': 'VARCHAR'}},
      auto_detect=True
    )
  ),

  medicare_nm as (
    SELECT *
    FROM medicare
    WHERE State = 'NM'
  ),

  medicare_abq as (
    SELECT *
    FROM medicare_nm
    WHERE "City/Town" IN ['ALBUQERQUE', 'RIO RANCHO']
  )

  SELECT *
  FROM medicare_abq
  ORDER BY "City/Town", NPI
  LIMIT 10
""").show()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬─────────────────┬────────────────────┬─────────────────────┬──────────────────────┬─────────┬─────────┬─────────┬─────────────────────────────────────────────────────┬────────┬───────────────────────────────────────────────┬────────────────────┬────────────┬────────────┬────────────┬────────────────────┬──────────┬──────────────────────────────────────────────────┬────────────┬─────────────┬──────────────────────────┬─────────────────────────────────────┬───────────┬────────────┬─────────┬───────────┬──────────────────┬───────────┬───────────┬───────────────────────────┐
│    NPI     │ Ind_PAC_ID │   Ind_enrl_ID   │ Provider Last Name │ Provider First Name │ Provider Middle Name │  suff   │  gndr   │  Cred   │                       Med_sch                       │ Grd_yr │                   pri_spec                    │     sec_spec_1     │ sec_spec_2 │ sec_spec_3 │ sec_spec_4 │    sec_spec_all    │ Telehlth │                  Facility Name                   

In [50]:
query = f"""
  SELECT *
  FROM read_csv(
    "{csv_file}",
    types={{'Telephone Number': 'VARCHAR'}},
    auto_detect=True
  )
""".strip()
print(query)


SELECT *
  FROM read_csv(
    "DAC_NationalDownloadableFile.csv",
    types={'Telephone Number': 'VARCHAR'}, 
    auto_detect=True
  )


In [51]:
rel = duckdb.sql(query)


In [54]:
export_query = '''
  COPY rel TO 'medicare_DAC_national'
  (FORMAT PARQUET, PARTITION_BY (State), OVERWRITE_OR_IGNORE 1);
'''
print(export_query)



  COPY rel TO 'medicare_DAC_national' 
  (FORMAT PARQUET, PARTITION_BY (State), OVERWRITE_OR_IGNORE 1);



In [55]:
duckdb.sql(export_query)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [56]:
!ls -la


total 814776
drwxr-xr-x  1 root root      4096 Jun 14 06:00 .
drwxr-xr-x  1 root root      4096 Jun 14 05:12 ..
drwxr-xr-x  4 root root      4096 Jun  4 13:32 .config
-rw-r--r--  1 root root 834304976 Jun 14 05:38 DAC_NationalDownloadableFile.csv
drwxr-xr-x 59 root root      4096 Jun 14 06:00 medicare_DAC_national
drwxr-xr-x  1 root root      4096 Jun  4 13:32 sample_data


In [59]:
!tree medicare_DAC_national/

medicare_DAC_national/
├── State=AK
│   └── data_0.parquet
├── State=AL
│   └── data_0.parquet
├── State=AR
│   └── data_0.parquet
├── State=AS
│   └── data_0.parquet
├── State=AZ
│   └── data_0.parquet
├── State=CA
│   └── data_0.parquet
├── State=CO
│   └── data_0.parquet
├── State=CT
│   └── data_0.parquet
├── State=DC
│   └── data_0.parquet
├── State=DE
│   └── data_0.parquet
├── State=FL
│   └── data_0.parquet
├── State=GA
│   └── data_0.parquet
├── State=GU
│   └── data_0.parquet
├── State=HI
│   └── data_0.parquet
├── State=IA
│   └── data_0.parquet
├── State=ID
│   └── data_0.parquet
├── State=IL
│   └── data_0.parquet
├── State=IN
│   └── data_0.parquet
├── State=KS
│   └── data_0.parquet
├── State=KY
│   └── data_0.parquet
├── State=LA
│   └── data_0.parquet
├── State=MA
│   └── data_0.parquet
├── State=MD
│   └── data_0.parquet
├── State=ME
│   └── data_0.parquet
├── State=MH
│   └── data_0.parquet
├── State=MI
│   └── data_0.parquet
├── State=MN
│   └── data_0.parquet
├── S

In [60]:
!du -ms medicare_DAC_national/


161	medicare_DAC_national/


In [61]:
dataset_path = "medicare_DAC_national/**/*.parquet"
dataset_path


'medicare_DAC_national/**/*.parquet'

In [62]:
query = f"SELECT * FROM read_parquet('{dataset_path}')"
query


"SELECT * FROM read_parquet('medicare_DAC_national/**/*.parquet')"

In [63]:
rel = duckdb.sql(query)


In [65]:
print(f"Total rows found across all partitions: {len(rel):,}")


Total rows found across all partitions: 3,364,553


In [66]:
rel.show()


┌────────────┬────────────┬─────────────────┬────────────────────┬─────────────────────┬──────────────────────┬─────────┬─────────┬─────────┬─────────────────────────────────────────────────┬────────┬───────────────────────────────────────────────┬───────────────────┬────────────┬────────────┬────────────┬───────────────────┬──────────┬─────────────────────────────────────────────┬────────────┬─────────────┬────────────────────┬────────────────────────────────────────────┬───────────┬────────────┬───────────┬──────────────────┬───────────┬───────────┬───────────────────────────┬─────────┐
│    NPI     │ Ind_PAC_ID │   Ind_enrl_ID   │ Provider Last Name │ Provider First Name │ Provider Middle Name │  suff   │  gndr   │  Cred   │                     Med_sch                     │ Grd_yr │                   pri_spec                    │    sec_spec_1     │ sec_spec_2 │ sec_spec_3 │ sec_spec_4 │   sec_spec_all    │ Telehlth │                Facility Name                │ org_pac_id │ num_or